# zReg — Basics Tutorial

This notebook introduces the core zReg workflow: loading a time-series of point clouds
from a `.tracklets` file, inspecting the per-frame data structure, and visualising two
frames side-by-side using an Open3D helper. It is designed as a first contact with the
library and requires no cluster access — just set `DATA_PATH` to your local file.

In [ ]:
DATA_PATH = "# Replace with the path to your .tracklets file"

import zreg
import copy
import open3d as o3d

## Load trajectory data

`load_data_from_tracklets` reads a `.tracklets` binary file and returns a list of
per-frame point cloud dictionaries. Each dictionary contains `"pos"` (3-D coordinates),
`"label"` (per-point label), and `"id"` (cell-track identifiers). The `device` argument
controls whether tensors are allocated on CPU or GPU.

In [ ]:
pcs, _alltracklets = zreg.dataset.load_data_from_tracklets(
    filepath=DATA_PATH,
    device="cpu",
)
print(pcs[0]["pos"].shape)
print(pcs[0]["label"].shape)
print(pcs[0]["id"].shape)

## Visualisation helper

`draw_registration_result` converts two point cloud dictionaries (or Open3D objects)
to legacy Open3D format, colours them gold and blue respectively, and renders them
together in an interactive Plotly viewer. This makes it straightforward to compare
a source and target frame before and after registration.

In [ ]:
def draw_registration_result(source, target):
    # deep copies used to avoid overwriting
    if isinstance(source, dict):
        # convert to open3d point cloud for visualization
        spc = zreg.dataset.torch_to_open3d(copy.deepcopy(source))
    else:
        spc = copy.deepcopy(source)
    if isinstance(target, dict):
        tpc = zreg.dataset.torch_to_open3d(copy.deepcopy(target))
    else:
        tpc = copy.deepcopy(target)
    spc.paint_uniform_color([1, 0.706, 0])
    tpc.paint_uniform_color([0, 0.651, 0.929])
    o3d.visualization.draw_plotly([spc.to_legacy(), tpc.to_legacy()])

## Visualise two frames

Here we display `pcs[0]` (the first frame) and `pcs[100]` (the 101st frame) of the
loaded trajectory. Rendering these two frames together gives a quick visual impression
of how much cell positions change over 100 time steps — a useful sanity check before
running any registration algorithm.

In [ ]:
draw_registration_result(pcs[0], pcs[100])